# Multi-class robustness on MNIST — and why the set representation matters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/fmaiv/blob/main/notebooks/06_day4_nn_mnist.ipynb)

**FMAIV Day 4 — frontier, part 2.** The companion notebook
[`05_day4_nn_robustness.ipynb`](https://colab.research.google.com/github/ttj/fmaiv/blob/main/notebooks/05_day4_nn_robustness.ipynb)
made the ideas concrete on a 2-D, 2-class toy. Here we scale to **MNIST** (784-D inputs, **10 classes**)
to show the two things the toy could only hint at:

1. **The decision is multi-class `argmax`.** Robustness at an input `x0` of true class `c` means the
   margin `z[c] - z[j] > 0` for **every** other class `j` over the whole L-infinity `eps`-ball — nine
   inequalities, all of which must hold.
2. **The set representation decides how much you can certify.** A verifier propagates an
   *over-approximation* of the reachable output set through the network; the looser that set, the more
   often the projected class scores overlap and certification fails (even when the network is robust).
   We compare three representations of increasing tightness — **IBP** (axis-aligned intervals) ⊂
   **CROWN** (linear bounds, akin to zonotopes) ⊂ **alpha-CROWN** (optimized linear bounds) — and watch
   the certified accuracy climb as the representation tightens.

Everything is **CPU-only** and runs in about a minute. This is the Python/auto_LiRPA view of the same
story our group's **NNV** toolbox tells with exact star-set / zonotope reachability — see the links at
the end.

## 0. Install (torch, torchvision, auto_LiRPA)
On Colab `torch`/`torchvision` are usually preinstalled; we add `auto_LiRPA` (pinned). All a no-op in the course Codespace.

In [ ]:
# Pin torch/torchvision to versions compatible with the pinned auto_LiRPA: newer
# torch removed torch.onnx.symbolic_helper._quantized_ops (which auto_LiRPA needs),
# so Colab's preinstalled newer torch breaks BoundedModule. Force the CPU pins.
!pip -q install torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cpu
!pip -q install matplotlib
!pip -q install git+https://github.com/Verified-Intelligence/auto_LiRPA.git@ca767f1d8c0a6b125a292ba165adb2319bbaf615
import torch; print('torch', torch.__version__)

## 1. MNIST and a small ReLU network
A compact MLP (784 -> 128 -> 64 -> 10), trained for a few epochs on CPU. Deterministic via a fixed seed.

In [ ]:
import torch, torch.nn as nn, torchvision
torch.manual_seed(0)

root = '/tmp/mnist'
tr = torchvision.datasets.MNIST(root, train=True,  download=True)
te = torchvision.datasets.MNIST(root, train=False, download=True)
Xtr = tr.data.float().div(255).unsqueeze(1); ytr = tr.targets   # (60000,1,28,28)
Xte = te.data.float().div(255).unsqueeze(1); yte = te.targets

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(),
                                 nn.Linear(784, 128), nn.ReLU(),
                                 nn.Linear(128, 64),  nn.ReLU(),
                                 nn.Linear(64, 10))
    def forward(self, x):
        return self.net(x)

model = Net()
opt = torch.optim.Adam(model.parameters(), 1e-3)
lossf = nn.CrossEntropyLoss()
for epoch in range(3):
    perm = torch.randperm(60000)
    for i in range(0, 60000, 256):
        idx = perm[i:i+256]
        opt.zero_grad(); lossf(model(Xtr[idx]), ytr[idx]).backward(); opt.step()
model.eval()
acc = (model(Xte[:2000]).argmax(1) == yte[:2000]).float().mean().item()
print(f'test accuracy (2000 images): {acc:.3f}')

## 2. Robustness is multi-class: nine margins must all stay positive

For an input `x0` of true class `c`, the prediction is `argmax` over the 10 logits. It is **robust** over
the `eps`-ball iff, for **every** rival class `j`, the margin `z[c] - z[j]` stays `> 0` across the ball.
We encode those nine margins in a specification matrix `C` and ask auto_LiRPA (CROWN) for a **sound lower
bound** on each. The image is certified iff the **smallest** of the nine lower bounds is `> 0` — that
minimum is the margin to the *nearest rival class*.

In [ ]:
from auto_LiRPA import BoundedModule, BoundedTensor
from auto_LiRPA.perturbations import PerturbationLpNorm
import numpy as np, matplotlib.pyplot as plt

bounded = BoundedModule(model, torch.empty_like(Xte[:1]),
                        bound_opts={'optimize_bound_args': {'iteration': 5}})  # few iters: alpha-CROWN stays fast on CPU

def spec_matrix(y):
    # For each sample, 9 rows e_c - e_j (j != c): each row value = z[c] - z[j].
    n = y.shape[0]; C = torch.zeros(n, 9, 10)
    for i in range(n):
        c = int(y[i]); r = 0
        for j in range(10):
            if j != c:
                C[i, r, c] = 1.0; C[i, r, j] = -1.0; r += 1
    return C

i0, eps_demo = 0, 0.01
c = int(yte[i0]); others = [j for j in range(10) if j != c]
lb, _ = bounded.compute_bounds(
    x=(BoundedTensor(Xte[i0:i0+1], PerturbationLpNorm(norm=float('inf'), eps=eps_demo)),),
    C=spec_matrix(yte[i0:i0+1]), method='CROWN')
margins = lb[0].tolist()
certified = min(margins) > 0

fig, (a0, a1) = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw={'width_ratios': [1, 2]})
a0.imshow(Xte[i0, 0], cmap='gray'); a0.set_title(f'image #{i0}: true class {c}'); a0.axis('off')
a1.bar(range(9), margins, color=['#1a7f37' if v > 0 else '#c0392b' for v in margins])
a1.axhline(0, color='k', lw=1); a1.set_xticks(range(9)); a1.set_xticklabels(others)
a1.set_xlabel('rival class j'); a1.set_ylabel(f'CROWN lower bound on z[{c}]-z[j]')
a1.set_title(f'eps={eps_demo}: {"CERTIFIED" if certified else "NOT certified"} '
             f'(nearest-rival margin = {min(margins):.2f})')
plt.tight_layout(); plt.show()

## 3. Tighter set representation => more certified

Now the headline. We compute **certified accuracy** — the fraction of test images proven robust — at a
few radii, with three representations of the propagated output set, from loosest to tightest:

- **IBP** — axis-aligned **intervals** (one `[lo, hi]` per neuron). Cheapest, loosest.
- **CROWN** — **linear** lower/upper bounds (a zonotope-like relaxation). Much tighter.
- **alpha-CROWN** — CROWN with the relaxation slopes **optimized**. Tighter still.

All are **sound** (they never certify a non-robust image), so any difference is pure **incompleteness**:
a looser set over-approximates more, its projected class scores overlap sooner, and it gives up. Watch
IBP certify almost nothing while CROWN certifies most images at the same radius. (This is the
multi-class, MNIST-scale version of the toy's CROWN-vs-IBP margin plot.)

In [ ]:
n = 100                     # images to certify per setting (alpha-CROWN uses few optimizer iters, so this stays fast)
eps_list = [0.005, 0.01, 0.02]
methods = [('IBP', 'IBP (intervals)', '#8250df'),
           ('CROWN', 'CROWN (linear bounds)', '#1f6feb'),
           ('CROWN-Optimized', 'alpha-CROWN (optimized)', '#1a7f37')]

C = spec_matrix(yte[:n])
clean_ok = model(Xte[:n]).argmax(1) == yte[:n]

def certified_accuracy(eps, method):
    lb, _ = bounded.compute_bounds(
        x=(BoundedTensor(Xte[:n], PerturbationLpNorm(norm=float('inf'), eps=eps)),),
        C=C, method=method)
    robust = (lb.min(1).values > 0) & clean_ok       # sound: certified AND actually correct
    return 100.0 * robust.float().mean().item()

rates = {m[0]: [certified_accuracy(e, m[0]) for e in eps_list] for m in methods}

fig, ax = plt.subplots(figsize=(6.5, 4)); x = np.arange(len(eps_list)); w = 0.26
for k, (key, label, col) in enumerate(methods):
    bars = ax.bar(x + (k - 1) * w, rates[key], w, label=label, color=col)
    ax.bar_label(bars, fmt='%.0f', fontsize=7, padding=1)   # label each bar (so 0% is explicit)
ax.set_xticks(x); ax.set_xticklabels([f'eps={e}' for e in eps_list])
ax.set_ylabel('certified accuracy (%)'); ax.set_ylim(0, 100)
ax.set_title(f'Certified accuracy on {n} MNIST images:\ntighter set representation => more certified', fontsize=10)
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()
for key, label, _ in methods:
    print(f'{label:26s} ' + '  '.join(f'eps={e}: {r:5.1f}%' for e, r in zip(eps_list, rates[key])))

## 4. The output set before argmax: per-class intervals (verify_fc analog)

This is the NNV [`verify_fc.m`](https://github.com/verivital/nnv/blob/master/code/nnv/examples/Tutorial/NN/MNIST/verify_fc.m) view of the same picture: instead of reporting the *margin* to each rival class, report **the reachable output set per output neuron** — CROWN's `[lo_j, hi_j]` interval on every logit `z[j]`. Robustness then reduces to **one inequality on those intervals**:

> `lo[true_class] > max_{j != true_class} hi[j]`  =>  **CERTIFIED ROBUST**

i.e. the true class's *worst-case* score still beats every rival's *best-case* score. Visually, the true class's interval has to sit *strictly above* the highest rival interval.

Script form: [`day04/examples/nn/verify_fc.py`](https://github.com/ttj/fmaiv/blob/main/day04/examples/nn/verify_fc.py).

In [ ]:
i0 = 0
eps_demo = 0.01
c = int(yte[i0])
ptb = PerturbationLpNorm(norm=float('inf'), eps=eps_demo)
lb, ub = bounded.compute_bounds(
    x=(BoundedTensor(Xte[i0:i0+1], ptb),), method='CROWN')
lo = lb[0].tolist(); hi = ub[0].tolist()
rival_hi  = max(hi[j] for j in range(10) if j != c)
rival_arg = max((j for j in range(10) if j != c), key=lambda j: hi[j])
robust = lo[c] > rival_hi

fig, (axI, axB) = plt.subplots(1, 2, figsize=(11, 4.2),
                               gridspec_kw={'width_ratios': [1, 2]})
axI.imshow(Xte[i0, 0], cmap='gray'); axI.set_title(f'image #{i0}: true class {c}'); axI.axis('off')

for j in range(10):
    if j == c:
        col = '#1a7f37'                                      # true class -- green
    elif hi[j] >= lo[c]:
        col = '#c0392b'                                      # rival interval reaches the certified threshold
    else:
        col = '#666'
    axB.plot([lo[j], hi[j]], [j, j], lw=4.5, color=col, alpha=0.92)
    axB.scatter([(lo[j]+hi[j])/2], [j], s=14, color=col)
axB.axvline(lo[c],     color='#1a7f37', ls='--', lw=1.2, label=f'lo[true={c}] = {lo[c]:+.2f}')
axB.axvline(rival_hi,  color='#c0392b', ls='--', lw=1.2,
            label=f'max hi[rival] = {rival_hi:+.2f}  (class {rival_arg})')
axB.set_yticks(range(10)); axB.set_ylabel('output class j'); axB.set_xlabel('z[j] (logit)')
axB.set_title(f'verify_fc-style output set, eps={eps_demo} '
              f'-> {"CERTIFIED" if robust else "NOT certified"}')
axB.legend(loc='lower right', fontsize=8); axB.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"image #{i0}, true class {c}, eps = {eps_demo}")
print(f"{'class':>5} {'lo':>9} {'hi':>9} {'width':>9}")
for j in range(10):
    mark = '  <-- true' if j == c else ''
    print(f"{j:>5} {lo[j]:+9.3f} {hi[j]:+9.3f} {hi[j]-lo[j]:9.3f}{mark}")
print(f"\ntrue-class lower bound : lo[{c}] = {lo[c]:+.3f}")
print(f"nearest-rival upper bnd: hi[{rival_arg}] = {rival_hi:+.3f}")
print(f"verdict: {'CERTIFIED ROBUST' if robust else 'not certified by CROWN'}")

## 5. When it is not robust: find a counterexample

A certified bound proves robustness; to show *non*-robustness we **falsify** — search the `eps`-ball for
an input the network misclassifies. Projected gradient descent (PGD) pushes `x0` toward its nearest rival
class while staying within `eps` (and within the valid pixel range `[0, 1]`). At a large enough radius it
finds an **adversarial example**: visually almost the same digit, different prediction.

In [ ]:
def pgd_attack(x, c, eps, steps=80, lr=0.02):
    rivals = [j for j in range(10) if j != c]
    delta = torch.zeros_like(x, requires_grad=True)
    for _ in range(steps):
        out = model(x + delta)[0]
        loss = out[c] - out[rivals].max()            # minimize true-class margin
        grad = torch.autograd.grad(loss, delta)[0]
        adv = (x + (delta - lr * grad.sign())).clamp(0, 1)   # project to L-inf ball AND valid pixels
        delta = (adv - x).clamp(-eps, eps).detach().requires_grad_(True)
    return (x + delta).detach()

eps_adv = 0.12
for i in range(500):                                  # first correctly-classified image we can break
    if model(Xte[i:i+1]).argmax(1).item() != int(yte[i]):
        continue
    adv = pgd_attack(Xte[i:i+1], int(yte[i]), eps_adv)
    pred = model(adv).argmax(1).item()
    if pred != int(yte[i]):
        fig, axs = plt.subplots(1, 3, figsize=(8, 3))
        axs[0].imshow(Xte[i, 0], cmap='gray'); axs[0].set_title(f'original: predicted {int(yte[i])}'); axs[0].axis('off')
        axs[1].imshow((adv - Xte[i:i+1])[0, 0], cmap='bwr', vmin=-eps_adv, vmax=eps_adv)
        axs[1].set_title(f'perturbation (|.| <= {eps_adv})'); axs[1].axis('off')
        axs[2].imshow(adv[0, 0], cmap='gray'); axs[2].set_title(f'adversarial: predicted {pred}'); axs[2].axis('off')
        plt.tight_layout(); plt.show()
        print(f'image #{i}: true {int(yte[i])}  ->  PGD adversary predicted {pred}  within eps={eps_adv}')
        break

## Takeaways

- **Robustness is multi-class.** Certified iff the margin to *every* rival class stays positive over the
  ball; the binding constraint is the nearest rival.
- **Soundness vs. completeness, again.** Every method here is sound (never certifies a non-robust image);
  the gaps are incompleteness. A **falsifier** (PGD) gives the other side — a concrete counterexample.
- **The set representation is the lever.** Looser over-approximations (intervals / IBP) lose far more to
  incompleteness than tighter ones (linear bounds / CROWN, then alpha-CROWN). Complete verifiers add
  branch-and-bound on top (e.g. **alpha,beta-CROWN**, the VNN-COMP winner) to close the remaining gap.

### Going further — exact set-based reachability (NNV)
Our group's **NNV** toolbox does this with *set representations* directly — **star sets**, **ImageStars**,
and **zonotopes** — and visualizes the reachable output sets per class:

- NNV: <https://verivital.github.io/nnv/>
- Tutorials: <https://github.com/verivital/nnv/tree/master/code/nnv/examples/Tutorial>
- MNIST: <https://github.com/verivital/nnv/tree/master/code/nnv/examples/Tutorial/NN/MNIST>
- Comparing reachability methods (intervals vs zonotopes vs stars): <https://github.com/verivital/nnv/tree/master/code/nnv/examples/Tutorial/NN/compareReachability>

The picture is the same one this notebook draws: the tighter the set representation, the more you can
prove, at higher cost.